In [1]:
import os
import yaml

PROJECT_ROOT = os.getcwd()
PRETRAIN_DIR = os.path.join(PROJECT_ROOT, "2_preprocessed_datasets", "merged_pretrain")
FINETUNE_DIR = os.path.join(PROJECT_ROOT, "2_preprocessed_datasets", "bccd_finetune")

# 1. Pretrain YAML
pretrain_yaml = os.path.join(PROJECT_ROOT, "scripts", "pretrain_yolo11.yaml")
pretrain_config = {
    'path': PRETRAIN_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'nc': 3,
    'names': {0: 'RBC', 1: 'WBC', 2: 'Platelet'}
}
with open(pretrain_yaml, 'w') as f:
    yaml.dump(pretrain_config, f, default_flow_style=False)

# 2. Finetune YAML (Includes Internal Test Set)
finetune_yaml = os.path.join(PROJECT_ROOT, "scripts", "finetune_yolo11.yaml")
finetune_config = {
    'path': FINETUNE_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test', # CRITICAL for your paper benchmark
    'nc': 3,
    'names': {0: 'RBC', 1: 'WBC', 2: 'Platelet'}
}
with open(finetune_yaml, 'w') as f:
    yaml.dump(finetune_config, f, default_flow_style=False)

print("Custom YAMLs generated successfully!")

Custom YAMLs generated successfully!


In [2]:
from ultralytics import YOLO

print("Starting Stage 1: Pre-training custom YOLOv11...")

output_dir = os.path.join(PROJECT_ROOT, "3_models")

# Load standard YOLOv11 (Nano or Small is highly recommended for medical patches)
model = YOLO('yolo11n.pt') 

results_pretrain = model.train(
    data=pretrain_yaml,
    epochs=50,             # Pretraining needs fewer epochs
    patience=15,
    imgsz=640,             # Standard resolution to learn base features
    batch=16,              
    workers=4,
    project=output_dir,
    name='yolo11_stage1_pretrain',
    device=0
)
print("Stage 1 Pretraining Complete!")

Starting Stage 1: Pre-training custom YOLOv11...
New https://pypi.org/project/ultralytics/8.4.33 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.166 🚀 Python-3.12.3 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4070 SUPER, 12282MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/linux-hasan/projects/blood_cell_project/scripts/pretrain_yolo11.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo

train: Scanning /home/linux-hasan/projects/blood_cell_project/2_preprocessed_datasets/merged_pretrain/labels/train.cache... 1182 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1182/1182 [00:00<?, ?it/s]

train: /home/linux-hasan/projects/blood_cell_project/2_preprocessed_datasets/merged_pretrain/images/train/92129d1652e75c3892beb376df57d274.png: 1 duplicate labels removed
train: /home/linux-hasan/projects/blood_cell_project/2_preprocessed_datasets/merged_pretrain/images/train/a8581e04ac765bd3ad658cf8e530a6ea.png: 1 duplicate labels removed
train: /home/linux-hasan/projects/blood_cell_project/2_preprocessed_datasets/merged_pretrain/images/train/c255521a08b75da09373db44e7f099c5.png: 17 duplicate labels removed


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.1±0.0 ms, read: 38.3±23.0 MB/s, size: 40.7 KB)


val: Scanning /home/linux-hasan/projects/blood_cell_project/2_preprocessed_datasets/merged_pretrain/labels/val.cache... 312 images, 0 backgrounds, 0 corrupt: 100%|██████████| 312/312 [00:00<?, ?it/s]

...

      50/50       3.8G     0.5713     0.4677     0.9691        202        640: 100%|██████████| 74/74 [00:12<00:00,  6.11it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 10/10 [00:02<00:00,  3.70it/s]

                   all        312       4549      0.776      0.871      0.904      0.749



50 epochs completed in 0.219 hours.
Optimizer stripped from /home/linux-hasan/projects/blood_cell_project/3_models/yolo11_stage1_pretrain/weights/last.pt, 5.5MB
Optimizer stripped from /home/linux-hasan/projects/blood_cell_project/3_models/yolo11_stage1_pretrain/weights/best.pt, 5.5MB

Validating /home/linux-hasan/projects/blood_cell_project/3_models/yolo11_stage1_pretrain/weights/best.pt...
Ultralytics 8.3.166 🚀 Python-3.12.3 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4070 SUPER, 12282MiB)
YOLO11n summary (fused): 100 layers, 2,582,737 parameters, 0 gradients, 6.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 10/10 [00:04<00:00,  2.43it/s]


                   all        312       4549      0.776      0.871      0.904      0.749
                   RBC        307        957      0.602      0.822      0.796      0.643
                   WBC        312       3431      0.864      0.858      0.954      0.874
              Platelet         99        161      0.863      0.932      0.962      0.731
Speed: 0.1ms preprocess, 1.2ms inference, 0.0ms loss, 2.3ms postprocess per image
Results saved to /home/linux-hasan/projects/blood_cell_project/3_models/yolo11_stage1_pretrain
✅ Stage 1 Pretraining Complete!


In [3]:
print("Starting Stage 2: High-Res Customized Fine-Tuning...")

# Load the weights we JUST trained
pretrained_weights = os.path.join(output_dir, "yolo11_stage1_pretrain", "weights", "best.pt")
model_finetune = YOLO(pretrained_weights)

results_finetune = model_finetune.train(
    data=finetune_yaml,
    epochs=100,
    patience=25,
    imgsz=800,             # NOVELTY: High-Res fine-tuning for tiny RBCs
    batch=8,               
    workers=4,
    project=output_dir,
    name='yolo11_stage2_finetune',
    device=0,
    
    # NOVELTY: Surgical Hyperparameter Customization
    box=10.0,              # Heavily penalize bad bounding boxes (default is 7.5)
    cls=0.5,               # Less penalty on class (since WBC is already perfectly learned)
    lr0=0.001,             # Slower learning rate for fine-tuning
    lrf=0.01,
    mixup=0.0,             # Disabled mixup (prevents artificial RBC overlapping)
    mosaic=1.0,
    degrees=90.0,
    fliplr=0.5,
    flipud=0.5
)
print("Stage 2 Fine-Tuning Complete!")

Starting Stage 2: High-Res Customized Fine-Tuning...
New https://pypi.org/project/ultralytics/8.4.33 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.166 🚀 Python-3.12.3 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4070 SUPER, 12282MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=10.0, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/linux-hasan/projects/blood_cell_project/scripts/finetune_yolo11.yaml, degrees=90.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, mod

train: Scanning /home/linux-hasan/projects/blood_cell_project/2_preprocessed_datasets/bccd_finetune/labels/train.cache... 689 images, 0 backgrounds, 0 corrupt: 100%|██████████| 689/689 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 562.3±452.2 MB/s, size: 12.7 KB)


val: Scanning /home/linux-hasan/projects/blood_cell_project/2_preprocessed_datasets/bccd_finetune/labels/val.cache... 73 images, 0 backgrounds, 0 corrupt: 100%|██████████| 73/73 [00:00<?, ?it/s]


...

     87/100      2.62G      1.324     0.7442      1.292         24        800: 100%|██████████| 87/87 [00:07<00:00, 11.83it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:00<00:00, 11.39it/s]

                   all         73        967      0.873      0.878      0.924       0.65
EarlyStopping: Training stopped early as no improvement observed in last 25 epochs. Best results observed at epoch 62, best model saved as best.pt.
To update EarlyStopping(patience=25) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



87 epochs completed in 0.308 hours.
Optimizer stripped from /home/linux-hasan/projects/blood_cell_project/3_models/yolo11_stage2_finetune/weights/last.pt, 5.5MB
Optimizer stripped from /home/linux-hasan/projects/blood_cell_project/3_models/yolo11_stage2_finetune/weights/best.pt, 5.5MB

Validating /home/linux-hasan/projects/blood_cell_project/3_models/yolo11_stage2_finetune/weights/best.pt...
Ultralytics 8.3.166 🚀 Python-3.12.3 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4070 SUPER, 12282MiB)
YOLO11n summary (fused): 100 layers, 2,582,737 parameters, 0 gradients, 6.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:01<00:00,  3.93it/s]


                   all         73        967      0.845      0.941       0.93      0.663
                   RBC         72        819      0.745      0.902      0.895      0.645
                   WBC         71         72      0.965          1      0.977      0.815
              Platelet         42         76      0.823       0.92      0.919      0.529
Speed: 0.1ms preprocess, 1.0ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to /home/linux-hasan/projects/blood_cell_project/3_models/yolo11_stage2_finetune
Stage 2 Fine-Tuning Complete!


In [4]:
print("Running Rigorous Internal Benchmark...")

finetuned_weights = os.path.join(output_dir, "yolo11_stage2_finetune", "weights", "best.pt")
best_yolo = YOLO(finetuned_weights)

# Evaluate on the strict internal test set
test_metrics = best_yolo.val(
    data=finetune_yaml,
    split='test',
    project=output_dir,
    name='yolo11_eval_internal_test',
    save_json=True,
    plots=True,
    
    # Validation Customizations to boost Precision
    conf=0.25,            # Ignore very weak predictions
    iou=0.60              # Stricter overlapping threshold
)

print(f"FINAL YOLOv11 mAP50: {test_metrics.box.map50:.4f}")
print("Validation plots saved!")

Running Rigorous Internal Benchmark...
Ultralytics 8.3.166 🚀 Python-3.12.3 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4070 SUPER, 12282MiB)
YOLO11n summary (fused): 100 layers, 2,582,737 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 944.1±356.4 MB/s, size: 16.1 KB)


val: Scanning /home/linux-hasan/projects/blood_cell_project/2_preprocessed_datasets/bccd_finetune/labels/test.cache... 76 images, 0 backgrounds, 0 corrupt: 100%|██████████| 76/76 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:02<00:00,  1.75it/s]


                   all         76       1021      0.823      0.939       0.92      0.673
                   RBC         72        873      0.717      0.898      0.869      0.649
                   WBC         74         74      0.961          1      0.992      0.854
              Platelet         40         74      0.792      0.919      0.901      0.516
Speed: 7.9ms preprocess, 5.0ms inference, 0.0ms loss, 4.0ms postprocess per image
Saving /home/linux-hasan/projects/blood_cell_project/3_models/yolo11_eval_internal_test/predictions.json...
Results saved to /home/linux-hasan/projects/blood_cell_project/3_models/yolo11_eval_internal_test
FINAL YOLOv11 mAP50: 0.9205
Validation plots saved!


In [5]:
print("Running External Validation on wbc_test...")

EXTERNAL_TEST_DIR = os.path.join(PROJECT_ROOT, "2_preprocessed_datasets", "wbc_test", "images", "val")

if os.path.exists(EXTERNAL_TEST_DIR):
    external_results = best_yolo.predict(
        source=EXTERNAL_TEST_DIR,
        conf=0.40,            
        iou=0.50,             
        project=output_dir,
        name='yolo11_external_inference',
        save=True,            
        save_txt=True         
    )
    print("External Testing complete. Visuals saved!")
else:
    print(f"Error: External directory not found at {EXTERNAL_TEST_DIR}")

Running External Validation on wbc_test...

WARNING ⚠️ 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/1931 /home/linux-hasan/projects/blood_cell_project/2_preprocessed_datasets/wbc_test/images/val/20190526_163027_0_jpg.rf.943b0a8a4e56431ce7ec29fb97ea5227.jpg: 800x800 1 RBC, 1 WBC, 5.7ms
image 2/1931 /home/linux-hasan/projects/blood_cell_project/2_preprocessed_datasets/wbc_test/images/val/20190526_163055_0_jpg.rf.6e86e24d25a3bdb777b3145a3314ddaa.jpg: 800x800 2 WBCs, 7.3ms
image 3/1931 /home/linux-hasan/

In [12]:
import os
import yaml
from ultralytics import YOLO

PROJECT_ROOT = os.getcwd()
EXTERNAL_TEST_DIR = os.path.join(PROJECT_ROOT, "2_preprocessed_datasets", "wbc_test")
output_dir = os.path.join(PROJECT_ROOT, "3_models")
finetuned_weights = os.path.join(output_dir, "yolo11_stage2_finetune", "weights", "best.pt")

print("Running External Validation to get exact mAP...")

# 1. Dynamically create a temporary YAML for the external dataset
external_yaml_path = os.path.join(PROJECT_ROOT, "scripts", "external_eval.yaml")
external_config = {
    'path': EXTERNAL_TEST_DIR,
    'train': 'images/val', # THE FIX: A dummy train path to satisfy Ultralytics
    'val': 'images/val',   # The actual validation images
    'nc': 3,
    'names': {0: 'RBC', 1: 'WBC', 2: 'Platelet'}
}

with open(external_yaml_path, 'w') as f:
    yaml.dump(external_config, f, default_flow_style=False)

# 2. Run the evaluator
best_yolo = YOLO(finetuned_weights)

try:
    external_metrics = best_yolo.val(
        data=external_yaml_path,
        split='val',          
        project=output_dir,
        name='yolo11_eval_external_mAP',
        save_json=True,
        plots=True,
        conf=0.25,
        iou=0.60
    )
    print(f"\nEXTERNAL Validation mAP50: {external_metrics.box.map50:.4f}")
except Exception as e:
    print(f"Error during evaluation: {e}")

Running External Validation to get exact mAP...
Ultralytics 8.3.166 🚀 Python-3.12.3 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4070 SUPER, 12282MiB)
YOLO11n summary (fused): 100 layers, 2,582,737 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.4±0.3 ms, read: 47.0±9.0 MB/s, size: 25.2 KB)


val: Scanning /home/linux-hasan/projects/blood_cell_project/2_preprocessed_datasets/wbc_test/labels/val... 1931 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1931/1931 [00:01<00:00, 1069.59it/s]


val: New cache created: /home/linux-hasan/projects/blood_cell_project/2_preprocessed_datasets/wbc_test/labels/val.cache


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 121/121 [00:13<00:00,  8.92it/s]


                   all       1931       1958      0.816      0.788       0.85      0.557
                   WBC       1931       1958      0.816      0.788       0.85      0.557
Speed: 0.3ms preprocess, 1.8ms inference, 0.0ms loss, 1.1ms postprocess per image
Saving /home/linux-hasan/projects/blood_cell_project/3_models/yolo11_eval_external_mAP4/predictions.json...
Results saved to /home/linux-hasan/projects/blood_cell_project/3_models/yolo11_eval_external_mAP4

EXTERNAL Validation mAP50: 0.8499


# Improvement trail
### Step 1: The 1-Minute Hack (TTA - Test Time)

In [ ]:
import os
from ultralytics import YOLO

PROJECT_ROOT = os.getcwd()
output_dir = os.path.join(PROJECT_ROOT, "3_models")
finetune_yaml = os.path.join(PROJECT_ROOT, "scripts", "finetune_yolo11.yaml")
finetuned_weights = os.path.join(output_dir, "yolo11_stage2_finetune", "weights", "best.pt")

print("Running Validation with TTA (Test Time Augmentation)...")
best_yolo = YOLO(finetuned_weights)

metrics_tta = best_yolo.val(
    data=finetune_yaml,
    split='test',
    project=output_dir,
    name='yolo11_eval_TTA',
    augment=True,         # THE HACK: Test Time Augmentation
    conf=0.30,            # Slightly stricter confidence
    iou=0.65              # Stricter overlapping check
)
print(f"New mAP50 with TTA: {metrics_tta.box.map50:.4f}")

Running Validation with TTA (Test Time Augmentation)...
Ultralytics 8.3.166 🚀 Python-3.12.3 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4070 SUPER, 12282MiB)
YOLO11n summary (fused): 100 layers, 2,582,737 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 53.6±12.7 MB/s, size: 14.5 KB)


val: Scanning /home/linux-hasan/projects/blood_cell_project/2_preprocessed_datasets/bccd_finetune/labels/test.cache... 76 images, 0 backgrounds, 0 corrupt: 100%|██████████| 76/76 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:02<00:00,  1.77it/s]


                   all         76       1021      0.849      0.918      0.929      0.669
                   RBC         72        873      0.739      0.889      0.892      0.668
                   WBC         74         74      0.968          1      0.993       0.84
              Platelet         40         74      0.841      0.865      0.903      0.498
Speed: 9.1ms preprocess, 10.4ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to /home/linux-hasan/projects/blood_cell_project/3_models/yolo11_eval_TTA3
New mAP50 with TTA: 0.9293


### Step 2: The Final Push (SAHI with YOLOv11)

In [13]:
import os
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
import json

PROJECT_ROOT = os.getcwd()
FINETUNE_DIR = os.path.join(PROJECT_ROOT, "2_preprocessed_datasets", "bccd_finetune")
best_weights = os.path.join(PROJECT_ROOT, "3_models", "yolo11_stage2_finetune", "weights", "best.pt")

print("Initializing SAHI with YOLOv11...")

# Load the model via SAHI (Using Ultralytics backend)
detection_model = AutoDetectionModel.from_pretrained(
    model_type='ultralytics',
    model_path=best_weights,
    confidence_threshold=0.35, 
    device="cuda:0", 
)

# Setup Directories
test_images_dir = os.path.join(FINETUNE_DIR, "images", "test")
sahi_pred_dir = os.path.join(PROJECT_ROOT, "3_models", "yolo11_eval_sahi", "labels")
os.makedirs(sahi_pred_dir, exist_ok=True)

# Delete old predictions
for f in os.listdir(sahi_pred_dir):
    os.remove(os.path.join(sahi_pred_dir, f))

image_files = [f for f in os.listdir(test_images_dir) if f.endswith(('.jpg', '.png'))]
print(f"Running YOLOv11 + SAHI Inference on {len(image_files)} test images...")

for img_name in image_files:
    img_path = os.path.join(test_images_dir, img_name)
    
    # SAHI MAGIC (Using 512x512 slices for YOLOv11 is often optimal)
    result = get_sliced_prediction(
        img_path,
        detection_model,
        slice_height=512,
        slice_width=512,
        overlap_height_ratio=0.2,
        overlap_width_ratio=0.2,
        postprocess_type="NMM", 
        postprocess_match_metric="IOU",
        postprocess_match_threshold=0.5
    )
    
    # Save predictions
    txt_name = os.path.splitext(img_name)[0] + '.txt'
    with open(os.path.join(sahi_pred_dir, txt_name), 'w') as f:
        for obj in result.object_prediction_list:
            box = obj.bbox
            img_w, img_h = result.image_width, result.image_height
            x_center = (box.minx + box.maxx) / 2 / img_w
            y_center = (box.miny + box.maxy) / 2 / img_h
            w = (box.maxx - box.minx) / img_w
            h = (box.maxy - box.miny) / img_h
            cls_id = obj.category.id
            conf = obj.score.value
            f.write(f"{cls_id} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f} {conf:.6f}\n")

print("YOLOv11 SAHI Inference Complete!")

Initializing SAHI with YOLOv11...
Running YOLOv11 + SAHI Inference on 76 test images...
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Performing prediction on 1 slices.
Pe

In [14]:
import os
import json
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

PROJECT_ROOT = os.getcwd()
FINETUNE_DIR = os.path.join(PROJECT_ROOT, "2_preprocessed_datasets", "bccd_finetune")

# Updated path for YOLOv11 SAHI predictions
sahi_pred_dir = os.path.join(PROJECT_ROOT, "3_models", "yolo11_eval_sahi", "labels")
test_images_dir = os.path.join(FINETUNE_DIR, "images", "test")
test_labels_dir = os.path.join(FINETUNE_DIR, "labels", "test")

print("Calculating Final mAP using COCOEval for YOLOv11+SAHI...")

categories = [
    {"id": 0, "name": "RBC"},
    {"id": 1, "name": "WBC"},
    {"id": 2, "name": "Platelet"}
]

coco_gt = {
    "info": {"description": "BCCD YOLOv11 SAHI Evaluation"}, 
    "images": [], 
    "annotations": [], 
    "categories": categories
}
coco_dt = [] # Predictions

img_files = [f for f in os.listdir(test_images_dir) if f.endswith(('.jpg', '.png'))]
ann_id_counter = 1

for img_id, img_name in enumerate(img_files):
    img_path = os.path.join(test_images_dir, img_name)
    base_name = os.path.splitext(img_name)[0]
    
    # Get image dimensions
    with Image.open(img_path) as img:
        img_w, img_h = img.size
        
    coco_gt["images"].append({
        "id": img_id, "file_name": img_name, "width": img_w, "height": img_h
    })
    
    # 1. Process Ground Truth (True Labels)
    gt_txt_path = os.path.join(test_labels_dir, base_name + '.txt')
    if os.path.exists(gt_txt_path):
        with open(gt_txt_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id, x_c, y_c, w, h = map(float, parts[:5])
                    abs_w, abs_h = w * img_w, h * img_h
                    x_min = (x_c * img_w) - (abs_w / 2)
                    y_min = (y_c * img_h) - (abs_h / 2)
                    
                    coco_gt["annotations"].append({
                        "id": ann_id_counter, "image_id": img_id, "category_id": int(cls_id),
                        "bbox": [x_min, y_min, abs_w, abs_h], "area": abs_w * abs_h, "iscrowd": 0
                    })
                    ann_id_counter += 1

    # 2. Process SAHI Predictions
    pred_txt_path = os.path.join(sahi_pred_dir, base_name + '.txt')
    if os.path.exists(pred_txt_path):
        with open(pred_txt_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 6:
                    cls_id, x_c, y_c, w, h, conf = map(float, parts[:6])
                    abs_w, abs_h = w * img_w, h * img_h
                    x_min = (x_c * img_w) - (abs_w / 2)
                    y_min = (y_c * img_h) - (abs_h / 2)
                    
                    coco_dt.append({
                        "image_id": img_id, "category_id": int(cls_id),
                        "bbox": [x_min, y_min, abs_w, abs_h], "score": conf
                    })

# Save the JSON files temporarily
gt_json_path = os.path.join(PROJECT_ROOT, "3_models", "yolo11_eval_sahi", "ground_truth.json")
dt_json_path = os.path.join(PROJECT_ROOT, "3_models", "yolo11_eval_sahi", "predictions.json")

with open(gt_json_path, 'w') as f:
    json.dump(coco_gt, f)
with open(dt_json_path, 'w') as f:
    json.dump(coco_dt, f)

# Initialize COCO API
cocoGt = COCO(gt_json_path)
cocoDt = cocoGt.loadRes(dt_json_path)

# Run COCO Evaluation
cocoEval = COCOeval(cocoGt, cocoDt, 'bbox')
cocoEval.evaluate()
cocoEval.accumulate()
cocoEval.summarize()

print("\n" + "="*40)
print("YOLOv11 + SAHI FINAL METRICS EXTRACTED:")
print("="*40)
print(f"Overall mAP50:      {cocoEval.stats[1]:.4f}")
print(f"Overall mAP50-95:   {cocoEval.stats[0]:.4f}")
print("="*40)

Calculating Final mAP using COCOEval for YOLOv11+SAHI...
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.13s).
Accumulating evaluation results...
DONE (t=0.02s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.557
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.840
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.664
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.158
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.551
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.647
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.388
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.596
 Average

# Figs

In [ ]:
import os
import shutil
from ultralytics import YOLO

PROJECT_ROOT = os.getcwd()
output_dir = os.path.join(PROJECT_ROOT, "3_models")
eval_dir = os.path.join(output_dir, "yolo11_eval_internal_test")
test_images_dir = os.path.join(PROJECT_ROOT, "2_preprocessed_datasets", "bccd_finetune", "images", "test")
best_weights = os.path.join(output_dir, "yolo11_stage2_finetune", "weights", "best.pt")

# Paper-er jonno ekta dedicated folder toiri kora
paper_dir = os.path.join(output_dir, "paper_visuals")
os.makedirs(paper_dir, exist_ok=True)

print("Preparing High-Quality Visuals for the Research Paper...")

# --- Step 1: Extract Core Graphs (Confusion Matrix & PR Curve) ---
print("Extracting evaluation graphs...")
graphs_to_extract = [
    'confusion_matrix_normalized.png', # Normalized is preferred for papers
    'PR_curve.png',                    # Precision-Recall curve is a MUST
    'F1_curve.png',                    # Shows the optimal confidence threshold
    'results.png'                      # Training loss curves
]

graphs_found = 0
for graph in graphs_to_extract:
    src_path = os.path.join(eval_dir, graph)
    dest_path = os.path.join(paper_dir, graph)
    if os.path.exists(src_path):
        shutil.copy(src_path, dest_path)
        graphs_found += 1
    else:
        print(f"Warning: Could not find {graph} in {eval_dir}")

print(f"Successfully copied {graphs_found} paper-ready graphs to 'paper_visuals'.")


# --- Step 2: Generate "Visually Stunning" Inference Images ---
print("🔬 Generating crisp bounding box predictions for the paper...")

model = YOLO(best_weights)

# Ekhane amra line_width=2 ebong conf=0.45 use korchi jate 
# chobite vul/halka box na thake, ekdom clean SOTA look ashe.
try:
    results = model.predict(
        source=test_images_dir,
        project=paper_dir,
        name='stunning_predictions',
        conf=0.45,            # Strict confidence for clean visuals
        iou=0.50,             # Keep bounding boxes distinct
        save=True,            # Save the images
        line_width=2,         # Professional box thickness
        show_labels=True,     # Show RBC/WBC text
        show_conf=False       # Hiding exact confidence % looks cleaner in papers
    )
    print("High-quality prediction images saved in 'paper_visuals/stunning_predictions'.")
except Exception as e:
    print(f"Error generating images: {e}")

print("\nAll visuals are ready! Please check the '3_models/paper_visuals' folder.")

Preparing High-Quality Visuals for the Research Paper...
Extracting evaluation graphs...
⚠️ Warning: Could not find PR_curve.png in /home/linux-hasan/projects/blood_cell_project/3_models/yolo11_eval_internal_test
⚠️ Warning: Could not find F1_curve.png in /home/linux-hasan/projects/blood_cell_project/3_models/yolo11_eval_internal_test
⚠️ Warning: Could not find results.png in /home/linux-hasan/projects/blood_cell_project/3_models/yolo11_eval_internal_test
Successfully copied 1 paper-ready graphs to 'paper_visuals'.
🔬 Generating crisp bounding box predictions for the paper...

image 1/76 /home/linux-hasan/projects/blood_cell_project/2_preprocessed_datasets/bccd_finetune/images/test/BloodImage_00001_jpg.rf.1a3206b15602db1d97193162a50bd001.jpg: 800x800 15 RBCs, 1 WBC, 5.8ms
image 2/76 /home/linux-hasan/projects/blood_cell_project/2_preprocessed_datasets/bccd_finetune/images/test/BloodImage_00003_jpg.rf.8d4037c47a4a76557c729057293e0755.jpg: 800x800 18 RBCs, 1 WBC, 6.0ms
image 3/76 /home/li